In [0]:
from pyspark.sql import functions as F


In [0]:
orders=spark.read.table('db_retail_analytics.supply_chain_project.silver_orders')
items=spark.read.table('db_retail_analytics.supply_chain_project.silver_order_items')

# brodcasting both tables to calculate the bussiness metrics and selecting the columns required for the powerbi
gold_fact_df=(
    orders
    .join(F.broadcast(items),on="order_id",how="inner")
    .withColumn("fulfillment_latency_days",F.datediff(F.col("order_delivered_customer_date"),F.col("order_purchase_timestamp")))
    .withColumn("is_delivery_on_time",
                F.when(F.col("order_delivered_customer_date")<=F.col("order_estimated_delivery_date"),1)
                .otherwise(0)
                )
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "seller_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "price",
        "freight_value",
        "fulfillment_latency_days",
        "is_delivery_on_time"
    )


)
#saving our gold table
(
    gold_fact_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.fact_sales_fulfillment")

)

print("fact_sales_fulfillment is saved as a table")

In [0]:
'''
we already have one gold_fact_table but for our next task we need to bring two silver tables geolocation and customers and join them with the gold_fact_table
'''

In [0]:
fact_sales=spark.read.table("db_retail_analytics.supply_chain_project.fact_sales_fulfillment")
customers=spark.read.table("db_retail_analytics.supply_chain_project.silver_customers")
geo=spark.read.table("db_retail_analytics.supply_chain_project.silver_geolocaiton")

'''
Now we need to join all the three tables
'''
enriched_gold_table = (
    fact_sales.alias("f")
    .join(
        customers.alias("c"),
        on="customer_id",
        how="left"
    )
    .join(
        geo.alias("g"),
        F.col("c.customer_zip_code_prefix") ==
        F.col("g.geolocation_zip_code_prefix"),
        how="left"
    )
    .select(
        F.col("f.*"),
        F.col("g.latitude"),
        F.col("g.longitude"),
        F.col("g.city"),
        F.col("g.state")
    )
)

(
    enriched_gold_table.
    write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") # for wiping out our old schema and replacing it with new schema if not we would get a [delta_metadata_mismatch error]
    .saveAsTable("db_retail_analytics.supply_chain_project.fact_sales_fulfillment")
)

In [0]:
'''
Beause I accidentally created an enriched gold table which would not be connected to the power bi, so without breaking our actual pipeline i would like to delte the accidental one and run the above code again with the original gold table
'''
spark.sql("DROP TABLE IF EXISTS db_retail_analytics.supply_chain_project.gold_enriched_sales")

In [0]:
'''
So, as our per our previous gold fact table(fact_sales_fulfillment) the dashboards are getting delayed/frozen in some cases, to over come this we need to optimize our approach according to our visuals we are going to build. The main idea is to pre aggregate the data and save in a table and in power Bi model view we create a realtionship (start schema model) 
'''
'''
Currently we are using direct query instead of import in powerbi. So, everytime you to examine a specific visual million of rows run through the specific query which not only slows down the visual response but also our compute get pricy $$$, at the end of the day we need to think in a cost effective manner. so we use a HYBRID/COMPOSITE approach, which would used our system RAM + only specific necessary compute. 
'''

In [0]:
orders=spark.read.table("db_retail_analytics.supply_chain_project.silver_orders")
items=spark.read.table("db_retail_analytics.supply_chain_project.silver_order_items")
customers=spark.read.table("db_retail_analytics.supply_chain_project.silver_customers")

'''
Now we need to join all the three tables
'''
# the re-engineered light weight star schema fact table
optimized_fact_df = (
    orders.alias("o")
    .join(
        F.broadcast(items).alias("i"),
        on="order_id",
        how="inner"
    )
    .join(
        F.broadcast(customers).alias("c"),
        on="customer_id",
        how="inner"
    )
    #adding the core metrics
    .withColumn("fulfillment_latency_days",F.datediff(F.col("order_delivered_customer_date"),F.col("order_purchase_timestamp")))
    .withColumn("is_delivery_on_time",F.when(F.col("order_delivered_customer_date")<=F.col("order_estimated_delivery_date"),1).otherwise(0))
    #selecting only the key metrics from now
    .select(
        F.col("o.order_id"),
        F.col("c.customer_id"),
        F.col("c.customer_zip_code_prefix"),
        F.col("i.product_id"),
        F.col("i.seller_id"),
        F.col("o.order_status"),
        F.col("o.order_purchase_timestamp"),
        F.col("i.price"),
        F.col("i.freight_value"),
        F.col("fulfillment_latency_days"),
        F.col("is_delivery_on_time")
    )
)

(
    optimized_fact_df.
    write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") # for wiping out our old schema and replacing it with new schema if not we would get a [delta_metadata_mismatch error]
    .saveAsTable("db_retail_analytics.supply_chain_project.fact_sales_fulfillment")
)

In [0]:
# creating a table just for the line chart
daily_summary_df=(
    optimized_fact_df
    #we only need date here so just extract the date
    .withColumn("purchase_date",F.to_date(F.col("order_purchase_timestamp")))
    .groupBy("purchase_date")
    .agg(
        F.avg("fulfillment_latency_days").alias("avg_fulfillment_latency"),
        F.count("order_id").alias("total_orders_volume")
    )
)
(
    daily_summary_df.
    write
    .format("delta").
    mode("overwrite").
    option("overwriteSchema", "true").
    saveAsTable("db_retail_analytics.supply_chain_project.gold_daily_logistics_summary")
)

In [0]:


spark.sql("""
CREATE OR REPLACE TABLE db_retail_analytics.supply_chain_project.gold_sellers_performance
CLUSTER BY(seller_id)
AS
SELECT
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    ROUND(SUM(oi.price),2) AS total_revenue,
    ROUND(AVG(r.review_score),2) AS avg_seller_review_score
FROM
db_retail_analytics.supply_chain_project.silver_order_items as oi
INNER JOIN
db_retail_analytics.supply_chain_project.silver_sellers AS s
    ON oi.seller_id=s.seller_id
LEFT JOIN db_retail_analytics.supply_chain_project.silver_reviews AS r
    ON oi.order_id=r.order_id
GROUP BY oi.seller_id,s.seller_city,s.seller_state

""")

In [0]:
'''
let's do some metrics for product categories 
'''

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE db_retail_analytics.supply_chain_project.gold_product_category_ranking
CLUSTER BY(product_category_name)
AS
WITH category_metrics AS(
    SELECT
        p.product_category_name,
        p.product_id,
        sum(oi.price) as product_revenue,
        avg(r.review_score) as avg_product_rating

    FROM
    db_retail_analytics.supply_chain_project.silver_order_items as oi
    INNER JOIN 
    db_retail_analytics.supply_chain_project.silver_products as p
        ON oi.product_id=p.product_id
    LEFT JOIN
    db_retail_analytics.supply_chain_project.silver_reviews as r
        ON oi.order_id=r.order_id
    GROUP BY  p.product_category_name,p.product_id
),ranked_categories AS(
    SELECT 
    *,
    SUM(product_revenue) OVER(PARTITION BY product_category_name) AS total_category_revenue,
    DENSE_RANK() over(order by SUM(product_revenue) OVER(PARTITION BY product_category_name) DESC) AS category_revenue_rank 
    FROM category_metrics
)
select
category_revenue_rank,
product_category_name,
product_id,
ROUND(product_revenue,2) as total_product_revenue,
ROUND(avg_product_rating,2) as avg_product_rating,
ROUND(total_category_revenue,2) as total_category_revenue
from ranked_categories
""")

In [0]:
'''
PHASE 2: GOLD TABLES (categorizing customeres in to threshold)
'''

In [0]:
spark.sql('''

CREATE OR REPLACE TABLE db_retail_analytics.supply_chain_project.gold_customer_installment_correlation
CLUSTER BY (price_threshold_tier) -- the new column we create in the below code
AS
SELECT
CASE 
WHEN payment_value <= 50 THEN "$0-$50"
WHEN payment_value <= 100 THEN "$0-$100"
WHEN payment_value <= 250 THEN "$0-$250"
WHEN payment_value <= 500 THEN "$0-$500"
ELSE "$500+"
END AS price_threshold_tier,
payment_type,
COUNT(DISTINCT order_id) AS total_orders,
ROUND(AVG(payment_installments),2) AS avg_installments,
ROUND(SUM(payment_value),2) AS total_financial_volume
FROM db_retail_analytics.supply_chain_project.silver_payments 
WHERE payment_type="credit_card"  -- currently we are only interested in credit card payments
GROUP BY price_threshold_tier, payment_type -- or you can write group by 1,2
''')

In [0]:
'''
Phase 3 : We are going to see the densly populated clusters of cusotomer region
'''

In [0]:
%sql
CREATE OR REPLACE TABLE db_retail_analytics.supply_chain_project.gold_geographic_revenue_density
CLUSTER BY (state,city)
AS
SELECT
SC.state,
SC.city,
COUNT(DISTINCT SO.order_id) AS total_orders,
COUNT(DISTINCT SC.customer_id) AS total_customers,
ROUND(SUM(SOI.price),2) AS total_revenue,
ROUND(SUM(SOI.price)/COUNT(DISTINCT SC.customer_id),2) AS revenue_per_customer,
ROUND(SUM(SOI.price)/COUNT(DISTINCT SO.order_id),2) AS revenue_per_order

FROM db_retail_analytics.supply_chain_project.silver_customers SC
INNER JOIN
db_retail_analytics.supply_chain_project.silver_orders SO
ON SC.customer_id = SO.customer_id
INNER JOIN 
db_retail_analytics.supply_chain_project.silver_order_items SOI
ON SO.order_id = SOI.order_id
GROUP BY SC.state,SC.city